In [1]:
### IMPORTS ###

# Install dependencies
%pip install anthropic python-dotenv

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


True

In [2]:
### CHAT SETUP ###

# Create an API client
from anthropic import Anthropic
client = Anthropic()
model = "claude-haiku-4-5"

# Helper functions to manage messages
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)
    
def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)
    
def chat(messages, system_prompt=None, temperature=0.7, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system_prompt:
        params["system"] = system_prompt
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [3]:
### CREATE DATASET WITH SOLUTION CRITERIA ###

import json  

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "python" | "json" | "regex",
    "solution_criteria": "Description of what constitutes a correct solution"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

dataset_with_solution_criteria = generate_dataset()
print(dataset_with_solution_criteria)

with open('dataset_with_solution_criteria.json', 'w') as f:
    json.dump(dataset_with_solution_criteria, f, indent=2)

[{'task': "Parse an AWS CloudFormation template and extract all resource logical IDs that are of type 'AWS::S3::Bucket'", 'format': 'python', 'solution_criteria': 'A function that takes a CloudFormation template (as a dict or JSON string) and returns a list of logical IDs for all S3 bucket resources. Should handle both JSON and YAML formats if provided as strings.'}, {'task': "Create a JSON policy document that allows an IAM user to read and list objects in a specific S3 bucket named 'my-data-bucket'", 'format': 'json', 'solution_criteria': 'A valid AWS IAM policy JSON object with appropriate actions (s3:GetObject, s3:ListBucket) and resources restricted to the specified bucket and its contents. Must be syntactically valid and follow AWS policy structure.'}, {'task': 'Create a regex pattern that matches valid AWS EC2 instance IDs (format: i-followed by 8 or 17 hexadecimal characters)', 'format': 'regex', 'solution_criteria': "A regex pattern that correctly matches EC2 instance IDs like

In [4]:
### GRADERS ### 

def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Original Task: 
    <task>{test_case['task']}</task>
    
    Solution to Evaluate: 
    <solution>{output}</solution>
    
    Criteria for Evaluation:
    <criteria>{test_case['solution_criteria']}</criteria>
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

import ast
import re

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    
def grade_by_syntax(test_case, output):
    """Grades the output based on the expected format"""
    expected_format = test_case["format"]
    
    if expected_format == "json":
        return validate_json(output)
    elif expected_format == "python":
        return validate_python(output)
    elif expected_format == "regex":
        return validate_regex(output)
    else:
        raise ValueError(f"Unknown format: {expected_format}")

In [5]:
### EVALUATION ###

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not include any explanations, comments or additional text.
"""
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    syntax_score = grade_by_syntax(test_case, output)
    score = (model_score + syntax_score) / 2
    
    return {
        "output": output, 
        "test_case": test_case, 
        "model_score": model_score,
        "syntax_score": syntax_score,
        "score": score,
        "reasoning": reasoning
    }
    
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

with open("dataset_with_solution_criteria.json", "r") as f:
    dataset_with_solution_criteria = json.load(f)

results = run_eval(dataset_with_solution_criteria)

print(json.dumps(results, indent=2))

Average score: 8.0
[
  {
    "output": "\nimport json\nimport sys\n\ndef extract_s3_buckets(template_str):\n    template = json.loads(template_str)\n    s3_bucket_ids = []\n    \n    resources = template.get('Resources', {})\n    for logical_id, resource in resources.items():\n        if resource.get('Type') == 'AWS::S3::Bucket':\n            s3_bucket_ids.append(logical_id)\n    \n    return s3_bucket_ids\n\nif __name__ == '__main__':\n    template_input = sys.stdin.read()\n    result = extract_s3_buckets(template_input)\n    print(json.dumps(result))\n",
    "test_case": {
      "task": "Parse an AWS CloudFormation template and extract all resource logical IDs that are of type 'AWS::S3::Bucket'",
      "format": "python",
      "solution_criteria": "A function that takes a CloudFormation template (as a dict or JSON string) and returns a list of logical IDs for all S3 bucket resources. Should handle both JSON and YAML formats if provided as strings."
    },
    "model_score": 5,
    "